# Demo 1 — Historical Data Download

This notebook downloads monthly Binance Spot Kline archives for the configured symbols, extracts the CSV files, and copies them into the external Volume.

It demonstrates:
- programmatic batch-source acquisition
- predictable URL construction
- ZIP download and extraction
- basic file validation
- idempotent overwrite behavior
- copying files from the driver into a Unity Catalog Volume

> Binance publishes public market data as daily and monthly archives. For Spot data from January 1, 2025 onward, timestamps are stored in microseconds.

## 1. Load shared configuration

The symbols, interval, month, and destination path are read from `config/00_config`.

In [0]:
%run ../config/00_config

## 2. Import required Python libraries

In [0]:
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.request import urlopen
from zipfile import BadZipFile, ZipFile

## 3. Define download settings

For this demo, the configuration produces URLs such as:

`https://data.binance.vision/data/spot/monthly/klines/BTCUSDT/1d/BTCUSDT-1d-2026-01.zip`

Files are temporarily stored on the Databricks driver before being copied to the external Volume.

In [0]:
binance_base_url = "https://data.binance.vision/data/spot/monthly/klines"
driver_download_dir = Path("/tmp/demo1_crypto_download")
driver_download_dir.mkdir(parents=True, exist_ok=True)

print(f"Symbols: {historical_symbols}")
print(f"Interval: {historical_interval}")
print(f"Period: {historical_period}")
print(f"Destination: {historical_raw_path}")

## 4. Build the expected download list

One monthly ZIP archive is downloaded for each configured symbol.

In [0]:
download_plan = []

for symbol in historical_symbols:
    archive_name = (
        f"{symbol}-{historical_interval}-{historical_period}.zip"
    )
    csv_name = (
        f"{symbol}-{historical_interval}-{historical_period}.csv"
    )
    url = (
        f"{binance_base_url}/{symbol}/{historical_interval}/"
        f"{archive_name}"
    )

    download_plan.append({
        "symbol": symbol,
        "archive_name": archive_name,
        "csv_name": csv_name,
        "url": url,
    })

display(spark.createDataFrame(download_plan))

In [0]:
%skip
local_files = [
    {"name": p.name, "path": str(p), "size_bytes": p.stat().st_size}
    for p in sorted(driver_download_dir.iterdir())
    if p.is_file()
]
display(spark.createDataFrame(local_files))

## 5. Download and extract the archives

The notebook validates that:
- the HTTP request succeeds
- the downloaded archive is not empty
- the ZIP can be opened
- the expected CSV exists inside the ZIP
- the extracted CSV is not empty

Existing local temporary files are overwritten, making this step safe to rerun.

In [0]:
download_results = []

for item in download_plan:
    symbol = item["symbol"]
    archive_path = driver_download_dir / item["archive_name"]
    csv_path = driver_download_dir / item["csv_name"]

    try:
        print(f"Downloading {item['url']}")

        with urlopen(item["url"], timeout=60) as response:
            archive_bytes = response.read()

        if not archive_bytes:
            raise RuntimeError(
                f"Downloaded archive is empty for {symbol}."
            )

        archive_path.write_bytes(archive_bytes)

        with ZipFile(archive_path, "r") as zip_file:
            zip_members = zip_file.namelist()

            if item["csv_name"] not in zip_members:
                raise FileNotFoundError(
                    f"Expected CSV {item['csv_name']} was not found "
                    f"inside {item['archive_name']}."
                )

            zip_file.extract(item["csv_name"], driver_download_dir)

        if not csv_path.exists() or csv_path.stat().st_size == 0:
            raise RuntimeError(
                f"Extracted CSV is missing or empty for {symbol}."
            )

        download_results.append({
            "symbol": symbol,
            "status": "DOWNLOADED",
            "csv_name": item["csv_name"],
            "size_bytes": csv_path.stat().st_size,
            "local_path": str(csv_path),
        })

    except HTTPError as exc:
        raise RuntimeError(
            f"Binance returned HTTP {exc.code} for {symbol}: {item['url']}"
        ) from exc
    except URLError as exc:
        raise RuntimeError(
            f"Could not reach Binance for {symbol}: {exc.reason}"
        ) from exc
    except BadZipFile as exc:
        raise RuntimeError(
            f"Downloaded file is not a valid ZIP archive for {symbol}."
        ) from exc

display(spark.createDataFrame(download_results))

## 6. Copy the CSV files into the external Volume

`dbutils.fs.cp` copies each extracted local driver file into:

`/Volumes/dbr_dev/parvinbadalov/demo1_crypto/raw/historical/`

`overwrite=True` keeps the notebook idempotent.

In [0]:
import shutil
from pathlib import Path

copy_results = []

for item in download_plan:
    local_csv_path = driver_download_dir / item["csv_name"]

    destination_path = (
        Path(historical_raw_path) / item["csv_name"]
    )

    shutil.copy2(
        local_csv_path,
        destination_path
    )

    copy_results.append({
        "symbol": item["symbol"],
        "status": "COPIED",
        "destination": str(destination_path),
    })

display(spark.createDataFrame(copy_results))

## 7. Validate the files in the Volume

The final check confirms that all expected CSV files exist in the destination folder and have non-zero sizes.

In [0]:
expected_csv_files = {
    item["csv_name"]
    for item in download_plan
}

volume_files = {
    file_info.name: file_info
    for file_info in dbutils.fs.ls(historical_raw_path)
    if file_info.name.lower().endswith(".csv")
}

missing_csv_files = expected_csv_files - set(volume_files)

if missing_csv_files:
    raise FileNotFoundError(
        f"Files missing from the Volume: {sorted(missing_csv_files)}"
    )

empty_csv_files = [
    file_name
    for file_name in expected_csv_files
    if volume_files[file_name].size == 0
]

if empty_csv_files:
    raise ValueError(
        f"Empty CSV files found in the Volume: {sorted(empty_csv_files)}"
    )

validation_rows = [
    {
        "file_name": file_name,
        "size_bytes": volume_files[file_name].size,
        "path": volume_files[file_name].path,
        "status": "READY",
    }
    for file_name in sorted(expected_csv_files)
]

display(spark.createDataFrame(validation_rows))
print("Historical data download completed successfully.")

## 8. Rerun behavior

Running this notebook again downloads and copies the same three files again. The destination file names stay unchanged, so the existing files are replaced rather than duplicated.

The next notebook is:

`batch/03_batch_ingestion.ipynb`